# 🔧 Gearbox Health — Cross-Turbine Comparison
**DSMLC Final Competition 2026 | Enbridge Wind Turbine SCADA Analysis**

### Research Question 1b
> *How do normal vs. anomalous periods differ across turbines, and what thresholds could trigger alerts?*

### Sections
1. Setup & Data Loading
2. Cross-Turbine Drift Comparison
3. Threshold Analysis — Does One Size Fit All?
4. Normal vs. Anomaly Distribution Comparison
5. Summary & Threshold Recommendations

---
## Section 1 — Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.figsize'] = (15, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('tab10')

print('✅ Imports OK')

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# ⚠️ UPDATE THIS PATH
BASE_PATH = Path('/Users/prabhaseessingh/Desktop/Data Science/DSMLCFinalComp/CARE_To_Compare/Wind Farm A')

DATASETS_PATH     = BASE_PATH / 'datasets'
EVENT_INFO_PATH   = BASE_PATH / 'event_info.csv'
FEATURE_DESC_PATH = BASE_PATH / 'feature_description.csv'

event_info = pd.read_csv(EVENT_INFO_PATH, sep=';')
feat_desc  = pd.read_csv(FEATURE_DESC_PATH, sep=';')

def get_label(bare_name):
    d = feat_desc[feat_desc['sensor_name'] == bare_name]['description'].values
    return d[0] if len(d) > 0 else bare_name

# Analysis parameters
DRIFT_WINDOW   = 1008  # 7-day rolling window (144 steps/day x 7)
BASELINE_STEPS = 4320  # first 30 days used as baseline
MULTIPLIER     = 2.3   # threshold = baseline mean + 2.3 * std
SENSOR_COL     = 'sensor_12_avg'  # Gearbox oil temperature
SENSOR_BARE    = 'sensor_12'

# All gearbox fault events in Farm A
GEARBOX_EVENT_IDS = [72, 10, 51]

# Normal events for comparison
normal_events = event_info[event_info['event_label'] == 'normal'].reset_index(drop=True)

print(f'Sensor: {get_label(SENSOR_BARE)} ({SENSOR_COL})')
print(f'Gearbox fault events to compare: {GEARBOX_EVENT_IDS}')
print(f'Normal events available: {len(normal_events)}')

---
## Section 2 — Cross-Turbine Drift Comparison
Load all 3 gearbox fault events and plot the 7-day rolling mean drift for each.
This shows whether different turbines exhibit the same pre-fault pattern.

In [ ]:
# Load all gearbox events and compute rolling means
event_data = {}

for event_id in GEARBOX_EVENT_IDS:
    df = pd.read_csv(DATASETS_PATH / f'{event_id}.csv', sep=';')
    df = df.sort_values('id').reset_index(drop=True)
    train_end = df[df['train_test'] == 'train'].index.max()
    rolling   = df[SENSOR_COL].rolling(
        window=DRIFT_WINDOW, min_periods=DRIFT_WINDOW//2
    ).mean()

    baseline  = rolling.iloc[:BASELINE_STEPS].dropna()
    base_mean = baseline.mean()
    base_std  = baseline.std()
    threshold = base_mean + MULTIPLIER * base_std

    pre_fault = rolling[rolling.index < train_end]
    exceeded  = pre_fault[pre_fault > threshold]

    if len(exceeded) > 0:
        first_idx  = exceeded.index[0]
        days_early = (train_end - first_idx) * 10 / 60 / 24
    else:
        first_idx  = None
        days_early = None

    info = event_info[event_info['event_id'] == event_id].iloc[0]
    event_data[event_id] = {
        'df': df, 'rolling': rolling, 'train_end': train_end,
        'base_mean': base_mean, 'base_std': base_std,
        'threshold': threshold, 'first_idx': first_idx,
        'days_early': days_early,
        'asset': info['asset'],
        'fault_type': info['event_description']
    }
    print(f'Event {event_id} | Turbine {info["asset"]} | {info["event_description"]}')
    print(f'  Baseline: {base_mean:.1f}°C | Threshold: {threshold:.1f}°C')
    if days_early:
        print(f'  First alert: {days_early:.1f} days before fault')
    else:
        print(f'  No drift detected at multiplier {MULTIPLIER}')
    print()

In [ ]:
# Plot all 3 events side by side
colors = ['steelblue', 'seagreen', 'darkorange']

fig, axes = plt.subplots(3, 1, figsize=(15, 13), sharex=False)
fig.suptitle(
    f'Gearbox Oil Temperature Drift — All 3 Fault Events (Farm A)\n'
    f'7-Day Rolling Mean | Threshold = Baseline Mean + {MULTIPLIER}σ',
    fontsize=13, fontweight='bold'
)

for ax, (event_id, d), color in zip(axes, event_data.items(), colors):
    # Raw signal
    ax.plot(d['df'].index, d['df'][SENSOR_COL],
            color=color, linewidth=0.4, alpha=0.2)
    # Rolling mean
    ax.plot(d['df'].index, d['rolling'],
            color=color, linewidth=2.2, label='7-day rolling mean')
    # Threshold
    ax.axhline(y=d['threshold'], color='red', linestyle='--', linewidth=1.5,
               label=f'Alert threshold ({d["threshold"]:.1f}°C)')
    # Fault window
    ax.axvline(x=d['train_end'], color='black', linestyle='--',
               linewidth=1.5, label='Fault window start')
    # First alert
    if d['first_idx'] is not None:
        ax.axvline(x=d['first_idx'], color='red', linewidth=1.5,
                   label=f"First alert ({d['days_early']:.1f} days early)")
        ax.axvspan(d['first_idx'], d['train_end'], alpha=0.08, color='red',
                   label='Warning period')

    ax.set_title(
        f"Event {event_id} — Turbine {d['asset']} | {d['fault_type']}",
        fontweight='bold', fontsize=10
    )
    ax.set_ylabel('°C')
    ax.set_xlabel('Time step (10-min intervals)')
    ax.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('cross_turbine_drift.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: cross_turbine_drift.png')

---
## Section 3 — Threshold Analysis: Does One Size Fit All?
If each turbine has a different baseline temperature, a fixed fleet-wide threshold
will miss faults on cool-running turbines or false-alarm on warm-running ones.
Here we test a range of multipliers across all 3 turbines.

In [ ]:
# Test multiple multipliers and count how many steps exceed threshold before fault
multipliers = [1.5, 1.8, 2.0, 2.3, 2.5, 3.0]

threshold_results = []

for event_id, d in event_data.items():
    rolling   = d['rolling']
    train_end = d['train_end']
    pre_fault = rolling[rolling.index < train_end]

    for m in multipliers:
        thresh   = d['base_mean'] + m * d['base_std']
        exceeded = pre_fault[pre_fault > thresh]

        if len(exceeded) > 0:
            first_idx  = exceeded.index[0]
            days_early = (train_end - first_idx) * 10 / 60 / 24
            false_pos_rate = len(exceeded) / len(pre_fault) * 100
        else:
            days_early     = None
            false_pos_rate = 0.0

        threshold_results.append({
            'Event': event_id,
            'Turbine': d['asset'],
            'Multiplier (σ)': m,
            'Threshold (°C)': round(thresh, 1),
            'Days Early': round(days_early, 1) if days_early else 'Not detected',
            '% Steps Flagged': round(false_pos_rate, 1)
        })

thresh_df = pd.DataFrame(threshold_results)
print('Threshold sensitivity across all gearbox events:')
thresh_df

In [ ]:
# Visualise: days early vs false positive rate (the precision/recall tradeoff)
fig, axes = plt.subplots(1, len(GEARBOX_EVENT_IDS), figsize=(16, 5), sharey=False)
fig.suptitle(
    'Threshold Sensitivity: Earliness vs. False Positive Rate\n'
    '(Lower multiplier = earlier warning but more false alarms)',
    fontsize=12, fontweight='bold'
)

for ax, (event_id, d), color in zip(axes, event_data.items(), colors):
    subset = thresh_df[thresh_df['Event'] == event_id].copy()
    # Only rows where drift was detected
    detected = subset[subset['Days Early'] != 'Not detected'].copy()
    detected['Days Early'] = detected['Days Early'].astype(float)

    ax2 = ax.twinx()

    ax.bar(detected['Multiplier (σ)'].astype(str), detected['Days Early'],
           color=color, alpha=0.6, label='Days early')
    ax2.plot(detected['Multiplier (σ)'].astype(str), detected['% Steps Flagged'],
             color='red', marker='o', linewidth=2, label='% steps flagged')

    ax.set_title(f"Turbine {d['asset']} | Event {event_id}", fontweight='bold', fontsize=9)
    ax.set_xlabel('Threshold multiplier (σ)')
    ax.set_ylabel('Days of early warning', color=color)
    ax2.set_ylabel('% timesteps flagged (false alarm proxy)', color='red')
    ax2.tick_params(axis='y', labelcolor='red')

    # Mark our chosen multiplier
    chosen_str = str(MULTIPLIER)
    if chosen_str in detected['Multiplier (σ)'].astype(str).values:
        idx = list(detected['Multiplier (σ)'].astype(str)).index(chosen_str)
        ax.get_children()[idx].set_edgecolor('black')
        ax.get_children()[idx].set_linewidth(2)

plt.tight_layout()
plt.savefig('threshold_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: threshold_sensitivity.png')

---
## Section 4 — Normal vs. Anomaly Distribution Comparison
Compare the statistical distribution of gearbox oil temp across
normal events and all 3 gearbox fault events using box plots and histograms.

In [ ]:
# Build a combined dataframe with labels for plotting
dist_frames = []

# Add 3 normal events
for _, row in normal_events.head(3).iterrows():
    df_n = pd.read_csv(DATASETS_PATH / f"{row['event_id']}.csv", sep=';')
    if SENSOR_COL in df_n.columns:
        dist_frames.append(pd.DataFrame({
            'temp': df_n[SENSOR_COL].dropna(),
            'label': f"Normal (Event {row['event_id']})",
            'type': 'Normal'
        }))

# Add gearbox fault events (training window only)
for event_id, d in event_data.items():
    train_data = d['df'][d['df']['train_test'] == 'train'][SENSOR_COL].dropna()
    dist_frames.append(pd.DataFrame({
        'temp': train_data,
        'label': f"Fault (Event {event_id}, Turbine {d['asset']})",
        'type': 'Pre-Fault'
    }))

dist_df = pd.concat(dist_frames, ignore_index=True)
print(f'Combined dataset: {len(dist_df):,} rows')
print(dist_df.groupby('type')['temp'].describe().round(2))

In [ ]:
# Box plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    'Gearbox Oil Temperature Distribution: Normal vs. Pre-Fault Events',
    fontsize=12, fontweight='bold'
)

# Box plot by event
palette = {'Normal': 'steelblue', 'Pre-Fault': 'tomato'}
sns.boxplot(
    data=dist_df, x='label', y='temp', hue='type',
    palette=palette, ax=ax1, dodge=False, legend=False
)
ax1.set_title('Distribution by Event', fontweight='bold')
ax1.set_xlabel('')
ax1.set_ylabel('Gearbox Oil Temperature (°C)')
ax1.tick_params(axis='x', rotation=30)

normal_patch = mpatches.Patch(color='steelblue', label='Normal')
fault_patch  = mpatches.Patch(color='tomato',    label='Pre-Fault')
ax1.legend(handles=[normal_patch, fault_patch], fontsize=9)

# Overlapping histogram — normal vs all fault events combined
normal_temps = dist_df[dist_df['type'] == 'Normal']['temp']
fault_temps  = dist_df[dist_df['type'] == 'Pre-Fault']['temp']

ax2.hist(normal_temps, bins=80, color='steelblue', alpha=0.5,
         density=True, label='Normal events')
ax2.hist(fault_temps,  bins=80, color='tomato',    alpha=0.5,
         density=True, label='Pre-fault events')

# Add vertical lines for means
ax2.axvline(normal_temps.mean(), color='steelblue', linewidth=2,
            linestyle='--', label=f'Normal mean ({normal_temps.mean():.1f}°C)')
ax2.axvline(fault_temps.mean(), color='tomato', linewidth=2,
            linestyle='--', label=f'Pre-fault mean ({fault_temps.mean():.1f}°C)')

ax2.set_title('Overall Temperature Distribution', fontweight='bold')
ax2.set_xlabel('Gearbox Oil Temperature (°C)')
ax2.set_ylabel('Density')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('gearbox_distribution_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: gearbox_distribution_comparison.png')

---
## Section 5 — Summary & Threshold Recommendations

In [ ]:
print('=' * 65)
print('CROSS-TURBINE GEARBOX ANALYSIS — KEY FINDINGS (Question 1b)')
print('=' * 65)

print('\n📊 Per-Turbine Baseline Temperatures (Gearbox Oil Temp):')
for event_id, d in event_data.items():
    print(f'   Turbine {d["asset"]} (Event {event_id}): '
          f'baseline = {d["base_mean"]:.1f}°C | '
          f'threshold = {d["threshold"]:.1f}°C')

print(f'\n   → Baseline temperatures differ across turbines.')
print(f'   → A fixed fleet-wide threshold would be inappropriate.')
print(f'   → Per-turbine baselines are required for reliable alerting.')

print('\n⏱️  Earliness Results at Multiplier 2.3σ:')
for event_id, d in event_data.items():
    if d['days_early']:
        print(f'   Turbine {d["asset"]} (Event {event_id}): {d["days_early"]:.1f} days early warning')
    else:
        print(f'   Turbine {d["asset"]} (Event {event_id}): drift not detected at 2.3σ')

print('\n✅ Recommended Alerting Strategy:')
print('   1. Compute a per-turbine baseline from first 30 days of operation')
print('   2. Set alert threshold at baseline mean + 2.3 standard deviations')
print('   3. Use 7-day rolling mean to smooth out daily temperature cycles')
print('   4. Trigger a maintenance flag when rolling mean exceeds threshold')
print('   5. Cross-validate with gearbox bearing temp (sensor_11) for confirmation')
print('   6. Re-baseline after major maintenance events')
print('=' * 65)